In [9]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

library(dplyr)
library(edgeR)
library(limma)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

In [2]:
mod_def <- "TopModPosBC"

In [ ]:
data_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected"

expr <- fread(paste0("data/", data_source, ".csv"), data.table=FALSE)

colnames(expr)[1] <- "Gene"

In [4]:
set_source <- "Claude_marker_genes"

In [ ]:
top_mods_df <- fread("data/enrichments/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules_Claude_marker_genes_enrichments.csv", data.table=FALSE)

top_mods_df$Cell_type <- gsub("/", "_", top_mods_df$Cell_type, fixed=TRUE)
top_mods_df$Cell_type <- gsub(" ", "_", top_mods_df$Cell_type, fixed=TRUE)

## Get modules and seed genes

In [6]:
top_mods_df %>%
    arrange(Qval) %>%
    select(Cell_type, Qval) %>%
    head(n=10)

,Cell_type,Qval
,<chr>,<dbl>
1,Micro_PVM,4.398842e-19
2,Astro,4.386429e-14
3,Oligo,5.042018e-14
4,All_GABAergic,1.023514e-13
5,VLMC,4.939285e-09
6,Sst,2.387589e-07
7,Endo,9.110287e-07
8,Vip,9.880391e-06
9,OPC,2.491505e-05


In [7]:
sets <- c("Micro_PVM", "Oligo", "Astro", "VLMC")

top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets,]

In [10]:
mod_genes_list <- lapply(seq_along(1:nrow(top_mods_df_subset)), function(i) {
        kME_path <- top_mods_df$kME_path[i]
        module <- top_mods_df$Module[i]
        get_mod_genes(kME_path, module, mod_def, n_genes=NULL)
    })
mod_genes <- unlist(mod_genes_list)

In [13]:
length(mod_genes)

[1] 3643

In [11]:
expr[1:5,1:5]

,Gene,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y,GTEX.11DXW.0011.R3b.SM.DNZZE
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,5S_rRNA,0.00000000,0.00000000,0.0004560151,0.00000000
2,5_8S_rRNA,0.00000000,0.00000000,0.0000000000,0.00000000
3,7SK,0.00000000,0.00000000,0.0000000000,0.00000000
4,A1BG,0.04672413,0.04198203,0.0441725843,0.06286158
5,A1BG-AS1,0.03827275,0.01159831,0.0182026263,0.01782835


In [12]:
expr_filtered <- expr[!expr[,1] %in% mod_genes,]

dim(expr_filtered)

[1] 51923   502

In [14]:
fwrite(expr_filtered, file=paste0("data/", data_source, "_", set_source, "_", paste(sets, collapse="_"), "_", mod_def, "_genes_removed.csv"))